In [ ]:
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

# Observed data
#trust_levels = np.array([23.0, 45.0, 67.0, 89.0, 34.0, 56.0, 78.0, 12.0, 90.0, 22.0]) / 100
#decisions = np.array([0, 0, 1, 1, 0, 0, 1, 0, 1, 0])


# Load the CSV file
file_path = '/Users/apple/Documents/Fall_2024_Research/1009_event_hardcoded_07_19_2024.csv'  # Update with your actual file path
data = pd.read_csv(file_path)

# Extract and scale the 'trust_level' column
trust_levels = data['trust_level'].values / 100  # Scale by dividing by 100

# Extract the 'move_approved' column
decisions = data['move_approved'].values

N = len(trust_levels)

with pm.Model() as model:
    # Hyperparameters
    sigma_theta = 0.05  # Standard deviation for theta evolution
    # Initial threshold with Beta(1,1) prior
    theta_0 = pm.Beta('theta_0', alpha=1, beta=1)

    # Develop a theta list (since theta_t is a time series variable)
    theta_list = [theta_0]

    # Evolving `theta_t` with constraints to stay within [0, 1]
    for t in range(1, N):
        # Propose a new `theta_t` with normal evolution, then map it to [0, 1] using the sigmoid
        theta_proposed = pm.Normal(f'theta_proposed_{t}', mu=theta_list[t-1], sigma=sigma_theta)
        theta_t = pm.Deterministic(f'theta_{t}', pm.math.sigmoid(theta_proposed))
        theta_list.append(theta_t)

    # Convert theta list to a deterministic variable
    theta = pm.Deterministic('theta', pm.math.stack(theta_list))


    # Compute decision probability using sigmoid function
    decision_prob = pm.Deterministic('decision_prob', pm.math.sigmoid(trust_levels - theta))

    # Observed decisions is bernoulli distribution using decision_prob as probability
    observed = pm.Bernoulli('observed', p=decision_prob, observed=decisions)

    # Inference
    # trace contains the collection of parameters of samples drawn from the posterior distributions 
    # 2000 samples to draw from for posterior
    # 1000 samples for warm-up iterations (but are discarded afterwards)
    # 90 percent of samples are accepted
    trace = pm.sample(draws=2000, tune=1000, target_accept=0.9, return_inferencedata=True)

# Trace plot
az.plot_trace(trace, var_names=['theta'])
plt.show()

# Summary statistics
print(az.summary(trace, var_names=['theta'], round_to=2))

# Trace plot
az.plot_trace(trace, var_names=['decision_prob'])
plt.show()


